# Experiment 3: Cross-Encoder Reranking + Statistical Significance Testing

This notebook is **self-contained** — it re-loads `docs.jsonl` and `qa_final_180.jsonl` and
rebuilds Dense/BM25/Hybrid from scratch (same as Experiment 2), then adds two new pieces:

1. **Cross-encoder reranking** on top of the best hybrid config (alpha=0.75): retrieve a
   top-10 candidate shortlist with hybrid search, then rerank those 10 with a proper
   query-document cross-encoder, and re-measure Recall@1/3/5 and MRR.
2. **Statistical significance testing**: is "hybrid beats dense" and "hybrid+rerank beats
   hybrid" a real effect, or could it be chance on 180 questions? Paired bootstrap
   confidence intervals + McNemar's test on the per-query correct/incorrect outcomes.

Run every cell top to bottom. Upload `docs.jsonl` and `qa_final_180.jsonl` when prompted.

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25

## Step 1: Upload data

In [ ]:
from google.colab import files
import os
os.makedirs('/content/data', exist_ok=True)
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, f'/content/data/{fname}')
print(os.listdir('/content/data'))

In [ ]:
import json, numpy as np

docs = [json.loads(l) for l in open('/content/data/docs.jsonl')]
qa   = [json.loads(l) for l in open('/content/data/qa_final_180.jsonl')]
texts = [d['text'] for d in docs]
ids   = [d['id'] for d in docs]
id_set = set(ids)
assert all(q['expected_doc_id'] in id_set for q in qa)
print(f"{len(docs)} docs, {len(qa)} questions")

## Step 2: Rebuild Dense, BM25, Hybrid (same formula as Experiment 2 — alpha=0.75 was the best config)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

def _norm(x):
    rng = x.max() - x.min()
    return (x - x.min()) / (rng + 1e-9)

def ranked_list_dense(query, k):
    q_emb = model.encode([query])
    _, idx = index.search(q_emb, k)
    return [ids[i] for i in idx[0]]

BEST_ALPHA = 0.75  # carried over from Experiment 2's alpha sweep result

def ranked_list_hybrid(query, k, alpha=BEST_ALPHA):
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))
    q_emb = model.encode([query])
    vec_scores = -np.linalg.norm(doc_embeddings - q_emb, axis=1)
    combined = alpha * _norm(vec_scores) + (1 - alpha) * _norm(bm25_scores)
    top = np.argsort(combined)[::-1][:k]
    return [ids[i] for i in top]

## Step 3: Cross-encoder reranking

`cross-encoder/ms-marco-MiniLM-L-6-v2` scores a (query, document) pair directly — it is
generally more accurate than embedding-similarity ranking, but much slower, so it is only
used to rerank a short candidate list, not the whole corpus.

Pipeline: hybrid (alpha=0.75) retrieves the **top-10** candidates -> cross-encoder scores
all 10 query-document pairs -> re-sort by cross-encoder score -> take the new top-1/3/5.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
doc_text_by_id = {d['id']: d['text'] for d in docs}

def ranked_list_hybrid_reranked(query, k, shortlist_size=10):
    shortlist = ranked_list_hybrid(query, shortlist_size)
    pairs = [(query, doc_text_by_id[doc_id]) for doc_id in shortlist]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1]
    reranked = [shortlist[i] for i in order]
    return reranked[:k]

## Step 4: Evaluation (same metric code as Experiment 2, but also returns a per-query 0/1 correctness array — needed for the significance tests in Step 6)

In [ ]:
def evaluate(rank_fn, k_max=5):
    per_cat = {c: {"n": 0, "r1": 0, "r3": 0, "r5": 0, "rr_sum": 0.0} for c in ["keyword", "semantic", "troubleshooting"]}
    overall = {"n": 0, "r1": 0, "r3": 0, "r5": 0, "rr_sum": 0.0}
    correct_at_1 = []   # per-query binary outcome, in the same order as `qa` -- for significance tests
    fails_at_3 = []

    for item in qa:
        full_rank = rank_fn(item['question'], k_max)
        target = item['expected_doc_id']
        cat = item['category']

        hit1 = target in full_rank[:1]
        hit3 = target in full_rank[:3]
        hit5 = target in full_rank[:5]
        rr = 1.0 / (full_rank.index(target) + 1) if target in full_rank else 0.0

        correct_at_1.append(int(hit1))

        for bucket in (overall, per_cat[cat]):
            bucket["n"] += 1
            bucket["r1"] += int(hit1)
            bucket["r3"] += int(hit3)
            bucket["r5"] += int(hit5)
            bucket["rr_sum"] += rr

        if not hit3:
            fails_at_3.append({"id": item['id'], "question": item['question'], "category": cat,
                                "expected_doc_id": target, "retrieved": full_rank})

    def summarize(b):
        return {"recall@1": b["r1"]/b["n"], "recall@3": b["r3"]/b["n"],
                "recall@5": b["r5"]/b["n"], "mrr": b["rr_sum"]/b["n"], "n": b["n"]}

    return {
        "overall": summarize(overall),
        "by_category": {c: summarize(b) for c, b in per_cat.items()},
        "correct_at_1": np.array(correct_at_1),   # length 180, 0/1
        "fails_at_3": fails_at_3,
    }

## Step 5: Run all four configurations

In [ ]:
dense_res   = evaluate(lambda q, k: ranked_list_dense(q, k))
hybrid_res  = evaluate(lambda q, k: ranked_list_hybrid(q, k))
rerank_res  = evaluate(lambda q, k: ranked_list_hybrid_reranked(q, k))

for name, res in [("Dense only", dense_res), ("Hybrid alpha=0.75", hybrid_res),
                   ("Hybrid + Cross-Encoder Rerank", rerank_res)]:
    print(f"{name}: {json.dumps(res['overall'], indent=2)}")

In [ ]:
import pandas as pd
summary_table = pd.DataFrame([
    {"method": "Dense only", **dense_res['overall']},
    {"method": "Hybrid alpha=0.75", **hybrid_res['overall']},
    {"method": "Hybrid + Rerank", **rerank_res['overall']},
])
summary_table

## Step 6: Statistical significance testing

Two complementary tests on the per-query Recall@1 outcomes (180 paired 0/1 values per method):

1. **Paired bootstrap** — resample the 180 questions with replacement 10,000 times,
   recompute the Recall@1 difference each time, and report the 95% confidence interval
   and the fraction of resamples where the difference is <= 0 (an empirical p-value-like
   quantity, one-sided).
2. **McNemar's test** — a standard test for paired binary outcomes; only looks at the
   questions where the two methods *disagree* (one right, one wrong), and tests whether
   the disagreements are lopsided in one direction.

Both are run for: (a) Hybrid vs. Dense-only, and (b) Hybrid+Rerank vs. Hybrid.

In [ ]:
def paired_bootstrap(a, b, n_boot=10000, seed=42):
    """a, b: same-length 0/1 arrays (correct_at_1 for method A and method B), paired by question.
    Returns: observed diff (mean(a) - mean(b)), 95% CI of the diff, and P(diff <= 0) under resampling."""
    rng = np.random.default_rng(seed)
    n = len(a)
    observed_diff = a.mean() - b.mean()
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = a[idx].mean() - b[idx].mean()
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_le_zero = (diffs <= 0).mean()
    return {"observed_diff": observed_diff, "ci_95": (ci_low, ci_high), "p_one_sided": p_le_zero}

def mcnemar(a, b):
    """a, b: same-length 0/1 arrays. Returns the 2x2 disagreement counts and McNemar's
    chi-square statistic (with continuity correction) + a rough p-value via scipy if available."""
    a_right_b_wrong = int(((a == 1) & (b == 0)).sum())
    a_wrong_b_right = int(((a == 0) & (b == 1)).sum())
    n_disagree = a_right_b_wrong + a_wrong_b_right
    if n_disagree == 0:
        return {"a_right_b_wrong": 0, "a_wrong_b_right": 0, "chi2": 0.0, "p_value": 1.0}
    chi2 = (abs(a_right_b_wrong - a_wrong_b_right) - 1) ** 2 / n_disagree
    try:
        from scipy.stats import chi2 as chi2_dist
        p_value = 1 - chi2_dist.cdf(chi2, df=1)
    except ImportError:
        p_value = None
    return {"a_right_b_wrong": a_right_b_wrong, "a_wrong_b_right": a_wrong_b_right,
            "chi2": chi2, "p_value": p_value}

print("=== Hybrid vs Dense-only ===")
boot_1 = paired_bootstrap(hybrid_res['correct_at_1'], dense_res['correct_at_1'])
mcn_1  = mcnemar(hybrid_res['correct_at_1'], dense_res['correct_at_1'])
print("Bootstrap:", boot_1)
print("McNemar:  ", mcn_1)

print()
print("=== Hybrid+Rerank vs Hybrid ===")
boot_2 = paired_bootstrap(rerank_res['correct_at_1'], hybrid_res['correct_at_1'])
mcn_2  = mcnemar(rerank_res['correct_at_1'], hybrid_res['correct_at_1'])
print("Bootstrap:", boot_2)
print("McNemar:  ", mcn_2)

**How to read this:**
- If the bootstrap 95% CI for `observed_diff` does **not** include 0, the difference is
  statistically significant at the 5% level.
- `p_one_sided` close to 0 means it's very unlikely the improvement is due to chance in
  this direction.
- McNemar's test with `p_value < 0.05` means the disagreements between the two methods
  are significantly lopsided (not just random noise), which is the direct paired-data
  equivalent of a significance test here.
- If either test comes back non-significant, **report that honestly** — "the
  improvement from reranking was not statistically significant at n=180" is a
  legitimate, common finding in small-scale RAG evaluations, and is more credible in a
  paper than an unqualified accuracy-only claim.

## Step 7: Category breakdown for Hybrid+Rerank (does reranking help most on 'semantic', the weakest category?)

In [ ]:
def category_table(res, name):
    rows = []
    for cat, m in res['by_category'].items():
        rows.append({"method": name, "category": cat, "Recall@1": m['recall@1'],
                     "Recall@3": m['recall@3'], "Recall@5": m['recall@5'], "MRR": m['mrr'], "n": m['n']})
    return rows

cat_rows = (category_table(dense_res, "Dense only") +
            category_table(hybrid_res, "Hybrid alpha=0.75") +
            category_table(rerank_res, "Hybrid + Rerank"))
cat_df = pd.DataFrame(cat_rows)
cat_df

## Step 8: Save everything for the paper

In [ ]:
exp3_export = {
    "summary": {
        "dense_only": dense_res['overall'],
        "hybrid_alpha_0.75": hybrid_res['overall'],
        "hybrid_plus_rerank": rerank_res['overall'],
    },
    "significance_hybrid_vs_dense": {
        "bootstrap": {"observed_diff": boot_1['observed_diff'],
                      "ci_95": list(boot_1['ci_95']), "p_one_sided": boot_1['p_one_sided']},
        "mcnemar": mcn_1,
    },
    "significance_rerank_vs_hybrid": {
        "bootstrap": {"observed_diff": boot_2['observed_diff'],
                      "ci_95": list(boot_2['ci_95']), "p_one_sided": boot_2['p_one_sided']},
        "mcnemar": mcn_2,
    },
}
with open('/content/experiment3_results.json', 'w') as f:
    json.dump(exp3_export, f, indent=2, default=float)

cat_df.to_csv('/content/experiment3_category_breakdown.csv', index=False)
summary_table.to_csv('/content/experiment3_summary.csv', index=False)

print("Saved: experiment3_results.json, experiment3_category_breakdown.csv, experiment3_summary.csv")
print("Download these three from the Colab Files panel and send them back.")

## What to report in the paper
- The **summary table** (Step 5) — Dense / Hybrid / Hybrid+Rerank side by side.
- The **significance results** (Step 6) — report both the bootstrap CI and McNemar's
  p-value for each comparison; this is what turns "76.1% vs 72.2%" into a defensible
  claim rather than just two numbers that happen to differ.
- The **category breakdown** (Step 7) — check specifically whether reranking's gain (if
  any) concentrates on the `semantic` category, which was the weakest one in Experiment 2.
- If reranking doesn't help, or isn't statistically significant, say so — this is a
  genuine possible outcome (cross-encoders are usually trained on general web-search
  data like MS MARCO, not Nginx-specific technical text, so a null result here is
  scientifically informative, not a failure).